In [1]:
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

In [2]:
filename = Path(r"C:/Users/karoa/MOHID_internship/MOHID_model_workflow/data/preprocessing/26442_55123_4020011_WIND_20230108144707_20230223144707.csv")
output_folder = filename.parent
# --- 1. Read the input file ---
df = pd.read_csv(
    filename,
    sep=r'\s+',
    skiprows=2,
    header=None,
    names=['year', 'month', 'day', 'hour', 'speed', 'direction'],
    na_values=['-9999.9'],
    engine='python',
)

df['datetime'] = pd.to_datetime(df[['year', 'month', 'day', 'hour']])

# --- 2. Convert speed + direction (meteorological) to X/Y components ---
dir_rad = np.deg2rad(df['direction'])
df['wind_x'] = -df['speed'] * np.sin(dir_rad)   # eastward component
df['wind_y'] = -df['speed'] * np.cos(dir_rad)   # northward component

# --- 3. Build the SECONDS column relative to the first timestamp ---
t0 = df['datetime'].iloc[0]
df['seconds'] = (df['datetime'] - t0).dt.total_seconds().astype(int)

# --- 4. Write the output file ---
# build fileout name
start = df['datetime'].iloc[0]
end = df['datetime'].iloc[-1]

fileout = output_folder / f"WIND_{start.year}_{start.month}_{start.day}_{end.year}_{end.month}_{end.day}.dat"
#fileout = f'WIND_{start.year}_{start.month}_{start.day}_{end.year}_{end.month}_{end.day}.dat'

initial = t0.strftime('%Y. %m. %d. %H. %M. %S.').replace(' 0', ' ')  
# Safer: build the header explicitly from the datetime parts
initial = f"{t0.year}. {t0.month:>2}. {t0.day:>2}. {t0.hour}. {t0.minute}. {t0.second}."

with open(fileout, 'w') as f:
    f.write("TIME_UNITS                : SECONDS\n")
    f.write(f"SERIE_INITIAL_DATA        : {initial}\n")
    f.write("\n")
    f.write("SECONDS                   Wind velocity X                 Wind velocity Y\n")
    f.write("<BeginTimeSerie>\n")
    for _, row in df.iterrows():
        # Skip rows where either component is NaN (from the -9999.9 fill values)
        if pd.isna(row['wind_x']) or pd.isna(row['wind_y']):
            continue
        f.write(f"{row['seconds']}                           "
                f"{row['wind_x']:.5g}                           "
                f"{row['wind_y']:.5g}\n")
    f.write("<EndTimeSerie>\n")